# ECG-SNN extra experiments (stronger evidence for the paper)
**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells **one by one, from top to bottom**.

| Part | Cells | What it gives the paper |
|---|---|---|
| Set-up | 1, 2, 4, 5 | Same data, models and training code as in your paper (nothing is changed). |
| **A** | 12 | Per-seed table, confusion matrices, McNemar vs A1 and ResNet-50, learned lead gates (needs your earlier `.pt` files, optional). |
| **B** | 13, 14 | 5-fold cross-validation on all 491 images, so every image is tested once (491 test predictions, not 71). |
| **C** | 15, 16, 17 | Training with rendered PTB-XL images added (Normal and MI) to see whether transfer improves. |
| Save | 18 | Zip of the small result files to download. |

Cells 13 and 16 are long. They can be run again after a stop: finished runs are skipped. **Report the numbers exactly as they come out.**


## Set-up (same as your Cells 1, 2, 4, 5)


In [ ]:
# Cell 1: Environment Setup & Cloud-to-Cloud Data Transfer
!pip -q install torch torchvision scikit-learn scipy pandas pillow matplotlib kagglehub

import os, glob, json, math, random, time, copy, hashlib
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms.functional as TF
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, cohen_kappa_score, matthews_corrcoef, f1_score
from scipy.stats import binomtest
import kagglehub

print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Active Device:", device)

# Download dataset directly into Colab's virtual drive
print("\nStreaming dataset from Kaggle...")
DATASET_PATH = kagglehub.dataset_download('kanishkarathore1604/ecg-image')
print("Dataset downloaded to:", DATASET_PATH)

# Automatically resolve the inner ECG_DATA folder path
if os.path.exists(os.path.join(DATASET_PATH, 'ECG_DATA')):
    DATA_ROOT = os.path.join(DATASET_PATH, 'ECG_DATA')
else:
    DATA_ROOT = DATASET_PATH
print("Verified Data Root:", DATA_ROOT)

WORK_DIR = '/content/work'
CACHE_DIR = os.path.join(WORK_DIR, 'cache')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

PyTorch Version: 2.14.1+cu130 | CUDA Available: True
Active Device: cuda

Streaming dataset from Kaggle...
Using Colab cache for faster access to the 'ecg-image' dataset.
Dataset downloaded to: /kaggle/input/ecg-image
Verified Data Root: /kaggle/input/ecg-image/ECG_DATA


In [ ]:
# Cell 2: Deduplication & 12-Lead Crop Caching (Forced Clean Re-cache)
import os, glob, hashlib, json
import numpy as np
from PIL import Image
from sklearn.model_selection import StratifiedKFold

# பழைய கேச் ஃபைல்களை நீக்குதல் (Force Clean)
for old_f in [os.path.join(CACHE_DIR, 'X.npy'), os.path.join(CACHE_DIR, 'meta.json')]:
    if os.path.exists(old_f):
        os.remove(old_f)

H, W = 96, 128
# Top 18% cuts text completely; Bottom 20% cuts bottom rhythm strip completely
margins = (0.03, 0.03, 0.18, 0.20)
valid_exts = ('.jpg', '.jpeg', '.png')

print("Scanning directory and deduplicating records...")
items_dict = {}

for p in sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.*"), recursive=True)):
    if p.lower().endswith(valid_exts):
        folder = os.path.basename(os.path.dirname(p))
        if 'myocardial infarction' in folder.lower() and 'history' not in folder.lower():
            label = 'Myocardial Infarction'
        elif 'history of mi' in folder.lower():
            label = 'History of MI'
        elif 'abnormal heartbeat' in folder.lower():
            label = 'Abnormal Heartbeat'
        elif 'normal' in folder.lower():
            label = 'Normal'
        else:
            continue

        with open(p, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()
        if file_hash not in items_dict:
            items_dict[file_hash] = (p, label)

clean_items = list(items_dict.values())
classes = sorted(list({c for _, c in clean_items}))
n_clean = len(clean_items)
print(f"✓ Unique records isolated: {n_clean} sheets across {len(classes)} classes: {classes}")

def crop_12_leads(im, margins=(0.03, 0.03, 0.18, 0.20)):
    w, h = im.size
    l, r, t, b = margins
    x0, x1 = int(w * l), int(w * (1.0 - r))
    y0, y1 = int(h * t), int(h * (1.0 - b))
    cw, ch = (x1 - x0) / 4.0, (y1 - y0) / 3.0
    crops = []
    for ri in range(3):
        for ci in range(4):
            box = (int(x0 + ci * cw), int(y0 + ri * ch), int(x0 + (ci + 1) * cw), int(y0 + (ri + 1) * ch))
            cropped = im.crop(box).resize((W, H), Image.BILINEAR)
            crops.append(np.asarray(cropped).transpose(2, 0, 1))
    return np.stack(crops)  # Shape: [12, 3, 96, 128]

X_cache = np.lib.format.open_memmap(os.path.join(CACHE_DIR, 'X.npy'), mode='w+', dtype=np.uint8, shape=(n_clean, 12, 3, H, W))
for idx, (path, _) in enumerate(clean_items):
    with Image.open(path) as im:
        X_cache[idx] = crop_12_leads(im.convert("RGB"), margins)
X_cache.flush()
print(f"✓ Freshly Re-cached {n_clean} sheets into {CACHE_DIR}/X.npy")

# Stratified Split (70% Train, 15% Val, 15% Test)
y_clean = np.array([classes.index(c) for _, c in clean_items])
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)
train_val_idx, test_idx = list(skf.split(np.zeros(n_clean), y_clean))[0]

skf_val = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)
tr_sub, val_sub = list(skf_val.split(np.zeros(len(train_val_idx)), y_clean[train_val_idx]))[0]

train_idx = train_val_idx[tr_sub]
val_idx = train_val_idx[val_sub]

meta_export = {
    "classes": classes,
    "paths": [p for p, _ in clean_items],
    "y": y_clean.tolist(),
    "train": train_idx.tolist(),
    "val": val_idx.tolist(),
    "test": test_idx.tolist(),
    "K": 12
}

with open(os.path.join(CACHE_DIR, 'meta.json'), 'w') as f:
    json.dump(meta_export, f, indent=2)

print(f"Partition Sizes -> Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Scanning directory and deduplicating records...
✓ Unique records isolated: 491 sheets across 4 classes: ['Abnormal Heartbeat', 'History of MI', 'Myocardial Infarction', 'Normal']
✓ Freshly Re-cached 491 sheets into /content/work/cache/X.npy
Partition Sizes -> Train: 350, Val: 70, Test: 71


In [ ]:
# Cell 4: Neural Network Architectures (Fully Corrected Lead Mappings)
class ATan(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha=2.0):
        ctx.save_for_backward(x); ctx.alpha = alpha
        return (x >= 0).to(x.dtype)
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors; a = ctx.alpha
        return g * (a / 2) / (1 + (math.pi / 2 * a * x) ** 2), None

class LIF(nn.Module):
    def __init__(self, beta=0.5, vth=1.0):
        super().__init__()
        self.vth = vth
        self.beta_logit = nn.Parameter(torch.tensor(math.log(beta / (1.0 - beta))))
        self.rate = None
        self.count = 0.0
    def forward(self, x):  # x: [T, Batch, Channels]
        b = torch.sigmoid(self.beta_logit)
        v = torch.zeros_like(x[0]); s = torch.zeros_like(x[0]); out = []
        for t in range(x.shape[0]):
            v = b * v + x[t] - s * self.vth
            s = ATan.apply(v - self.vth)
            out.append(s)
        o = torch.stack(out)
        self.rate = o.mean()
        self.count = o.detach().sum().item()
        return o

class ReLUAct(nn.Module):
    def __init__(self):
        super().__init__()
        self.rate = None; self.count = 0.0
    def forward(self, x): return F.relu(x)

def build_neuron(ann): return ReLUAct() if ann else LIF()

class TLinBN(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.lin = nn.Linear(i, o)
        self.bn = nn.BatchNorm1d(o)
    def forward(self, x):
        T, N, C = x.shape
        return self.bn(self.lin(x.reshape(T * N, C))).reshape(T, N, -1)

class Encoder(nn.Module):
    def __init__(self, mode, D=64, T=8, ann=False):
        super().__init__()
        self.mode, self.T = mode, T
        self.ap = nn.Parameter(torch.ones(1, 1, D))
        self.an = nn.Parameter(torch.ones(1, 1, D))
        self.neu = build_neuron(ann)
        self.out_dim = 2 * D if mode in ("polar_temporal", "polar_static") else D
        self.register_buffer("rms", torch.ones(1, D, 1))

    def forward(self, u):
        if self.training:
            r = u.detach().pow(2).mean(dim=(0, 2), keepdim=True).sqrt() + 1e-5
            self.rms.mul_(0.9).add_(0.1 * r)
        u = u / self.rms

        # Temporal column unrolling (Width = T steps)
        if self.mode in ("polar_temporal", "temporal_mag"):
            if u.shape[-1] > self.T: u = F.adaptive_avg_pool1d(u, self.T)
            elif u.shape[-1] < self.T: u = F.interpolate(u, size=self.T, mode="linear", align_corners=False)
            seq = u.permute(2, 0, 1)  # [T, Batch*12, Channels=64]
        else:
            seq = u.mean(-1).unsqueeze(0).expand(self.T, -1, -1)

        # Polarity decomposition
        if self.mode in ("polar_temporal", "polar_static"):
            cur = torch.cat([F.relu(seq) * self.ap, F.relu(-seq) * self.an], dim=-1)
        elif self.mode == "temporal_mag":
            cur = seq.abs() * self.ap
        else:
            cur = seq * self.ap
        return self.neu(cur)

class DenseHead(nn.Module):
    def __init__(self, in_dim, growth=32, L=4, ann=False):
        super().__init__()
        c = in_dim; lins = []
        for _ in range(L): lins.append(TLinBN(c, growth)); c += growth
        self.lins = nn.ModuleList(lins)
        self.neus = nn.ModuleList([build_neuron(ann) for _ in range(L)])
        self.out_dim = c

    def forward(self, x):
        c = x
        for lin, neu in zip(self.lins, self.neus):
            c = torch.cat([c, neu(lin(c))], -1)
        return c

class Fusion(nn.Module):
    def __init__(self, in_dim, K, n_cls, mode="lead_aware", ann=False):
        super().__init__()
        self.mode, self.K = mode, K
        # Correct Anatomical Lead Mapping:
        # ['I'(0), 'aVR'(1), 'V1'(2), 'V4'(3), 'II'(4), 'aVL'(5), 'V2'(6), 'V5'(7), 'III'(8), 'aVF'(9), 'V3'(10), 'V6'(11)]
        self.groups = [
            [4, 8, 9],       # Inferior: II, III, aVF
            [0, 5, 7, 11],   # Lateral: I, aVL, V5, V6
            [2, 6],          # Septal: V1, V2
            [10, 3],         # Anterior: V3, V4
            [1]              # aVR alone
        ] if K == 12 else [[0]]

        if mode == "lead_aware":
            self.gam = nn.Parameter(torch.zeros(K))
            self.gfc = nn.ModuleList([nn.Linear(in_dim, 64) for _ in self.groups])
            self.gneu = nn.ModuleList([build_neuron(ann) for _ in self.groups])
            self.fus = nn.Linear(64 * len(self.groups), 128)
        else:
            self.fus = nn.Linear(K * in_dim, 128)
        self.fneu = build_neuron(ann)
        self.out = nn.Linear(128, n_cls)

    def forward(self, c):
        T, B = c.shape[:2]
        if self.mode == "lead_aware":
            g = torch.sigmoid(self.gam)
            gs = []
            for fc, neu, idx in zip(self.gfc, self.gneu, self.groups):
                weighted_lead_sum = (c[:, :, idx, :] * g[idx][None, None, :, None]).sum(dim=2)
                gs.append(neu(fc(weighted_lead_sum)))
            z = torch.cat(gs, dim=-1)
        else:
            z = c.reshape(T, B, -1)
        return self.out(self.fneu(self.fus(z))).mean(0)

class SpikeECG(nn.Module):
    def __init__(self, n_cls, K=12, enc="polar_temporal", fusion="lead_aware", T=8, ann=False):
        super().__init__()
        self.K, self.T = K, T
        w = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1
        self.backbone = torchvision.models.mobilenet_v2(weights=w).features[:14]
        self.adapter = nn.Conv2d(96, 64, 1)
        self.enc = Encoder(enc, 64, T, ann)
        self.head = DenseHead(self.enc.out_dim, 32, 4, ann)
        self.fusion = Fusion(self.head.out_dim, K, n_cls, fusion, ann)

    def forward(self, x):
        B, K = x.shape[:2]
        u = self.adapter(self.backbone(x.flatten(0, 1))).mean(2)
        s = self.enc(u)
        c = self.head(s)
        return self.fusion(c.reshape(c.shape[0], B, K, -1))

    def neurons(self): return [m for m in self.modules() if isinstance(m, (LIF, ReLUAct))]
    def rate_loss(self):
        r = [m.rate for m in self.neurons() if m.rate is not None]
        return torch.stack(r).mean() if r else torch.tensor(0.0)

def mosaic(x):
    B, K, C, H, W = x.shape
    return x.view(B, 3, 4, C, H, W).permute(0, 3, 1, 4, 2, 5).reshape(B, C, 3 * H, 4 * W)

class MosaicNet(nn.Module):
    def __init__(self, kind, n_cls):
        super().__init__()
        tv = torchvision.models
        if kind == "resnet50":
            m = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V1)
            m.fc = nn.Linear(2048, n_cls)
        elif kind == "lwdn161":
            mb = tv.mobilenet_v2(weights=tv.MobileNet_V2_Weights.IMAGENET1K_V1).features[:14]
            dn = tv.densenet161(weights=tv.DenseNet161_Weights.IMAGENET1K_V1).features
            m = nn.Sequential(
                mb, nn.Conv2d(96, 192, 1),
                dn.denseblock2, dn.transition2,
                dn.denseblock3, dn.transition3,
                dn.denseblock4, dn.norm5,
                nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(2208, n_cls)
            )
        self.net = m
    def forward(self, x): return self.net(mosaic(x))

def build_model(model_id, n_cls, K=12):
    reg = {
        "S1": dict(enc="polar_temporal", fusion="lead_aware", T=8),
        "S2": dict(enc="direct", fusion="lead_aware", T=8),
        "S3": dict(enc="polar_static", fusion="lead_aware", T=8),
        "S4": dict(enc="temporal_mag", fusion="lead_aware", T=8),
        "S5": dict(enc="polar_temporal", fusion="concat", T=8),
        "S6a": dict(enc="polar_temporal", fusion="lead_aware", T=4),
        "S7": dict(enc="polar_temporal", fusion="lead_aware", T=8),
    }
    if model_id in reg: return SpikeECG(n_cls, K, ann=False, **reg[model_id]), True
    if model_id == "A4": return SpikeECG(n_cls, K, "polar_temporal", "lead_aware", 8, ann=True), False
    if model_id == "A1": return MosaicNet("lwdn161", n_cls), False
    if model_id == "A2r": return MosaicNet("resnet50", n_cls), False
    raise ValueError(model_id)

print("✓ Architectures registered with corrected territory mappings.")

✓ Architectures registered with corrected territory mappings.


In [ ]:
# Cell 5: Training & Neuromorphic Energy Accounting Engine
import copy, json, os, random
import numpy as np
from sklearn.metrics import (
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms.functional as TF

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
E_MAC, E_AC = 4.6e-12, 0.9e-12


def augment_training_crop(x, rng):
  if rng.random() < 0.8:
    x = (x * rng.uniform(0.85, 1.15)).clamp(0, 1)
  if rng.random() < 0.8:
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = ((x - m) * rng.uniform(0.85, 1.15) + m).clamp(0, 1)
  if rng.random() < 0.3:
    x = x[:, torch.randperm(3)]
  if rng.random() < 0.3:
    x = (x + torch.randn_like(x) * rng.uniform(0, 0.02)).clamp(0, 1)
  if rng.random() < 0.3:
    x = TF.gaussian_blur(x, kernel_size=5, sigma=rng.uniform(0.3, 1.0))
  return x


class ECGDataset(Dataset):

  def __init__(self, cache_dir, idx, train=False, seed=0):
    self.X = np.load(os.path.join(cache_dir, 'X.npy'), mmap_mode='r')
    self.meta = json.load(open(os.path.join(cache_dir, 'meta.json')))
    self.idx = np.array(idx)
    self.y = np.array(self.meta['y'])[self.idx]
    self.train = train
    self.seed = seed

  def __len__(self):
    return len(self.idx)

  def __getitem__(self, i):
    x = torch.from_numpy(np.array(self.X[self.idx[i]])).float() / 255.0
    if self.train:
      x = augment_training_crop(
          x, random.Random(self.seed * 10007 + i + random.randrange(10**5))
      )
    return (x - MEAN) / STD, int(self.y[i])


def compute_metrics(y, pred, n_cls):
  cm = confusion_matrix(y, pred, labels=range(n_cls))
  n = cm.sum()
  tp = np.diag(cm)
  fp = cm.sum(0) - tp
  fn = cm.sum(1) - tp
  tn = n - tp - fp - fn
  P = tp / np.maximum(tp + fp, 1)
  R = tp / np.maximum(tp + fn, 1)
  Sp = tn / np.maximum(tn + fp, 1)
  F1 = 2 * P * R / np.maximum(P + R, 1e-12)
  return dict(
      n=int(n),
      acc=float(tp.sum() / n),
      macro_f1=float(F1.mean()),
      kappa=float(cohen_kappa_score(y, pred)),
      mcc=float(matthews_corrcoef(y, pred)),
      precision=P.tolist(),
      sensitivity=R.tolist(),
      specificity=Sp.tolist(),
      f1=F1.tolist(),
      support=cm.sum(1).tolist(),
      confusion=cm.tolist(),
  )


@torch.no_grad()
def evaluate_loader(model, loader, dev):
  model.eval()
  ps, ys = [], []
  for x, y in loader:
    ps.append(F.softmax(model(x.to(dev)).float(), -1).cpu())
    ys.append(y)
  return torch.cat(ps).numpy(), torch.cat(ys).numpy()


def execute_train_run(
    model_id, cache_dir, out_dir, seed=0, epochs=35, bs=16, lr=1e-3
):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  meta = json.load(open(os.path.join(cache_dir, 'meta.json')))
  n_cls, K = len(meta['classes']), meta['K']

  tr = ECGDataset(cache_dir, meta['train'], train=True, seed=seed)
  va = ECGDataset(cache_dir, meta['val'], train=False)
  te = ECGDataset(cache_dir, meta['test'], train=False)

  cnt = np.bincount(tr.y, minlength=n_cls)
  w = (1.0 / np.maximum(cnt, 1))[tr.y]
  sampler = WeightedRandomSampler(
      torch.tensor(w, dtype=torch.double), len(tr), replacement=True
  )

  ltr = DataLoader(
      tr, batch_size=bs, sampler=sampler, num_workers=2, pin_memory=True
  )
  lva = DataLoader(va, batch_size=bs, shuffle=False, num_workers=2)
  lte = DataLoader(te, batch_size=bs, shuffle=False, num_workers=2)

  model, spiking = build_model(model_id, n_cls, K)
  model.to(device)

  if isinstance(model, SpikeECG):
    bb = list(model.backbone.parameters())
    ids = {id(p) for p in bb}
    groups = [
        {'params': bb, 'lr': 1e-4},
        {
            'params': [p for p in model.parameters() if id(p) not in ids],
            'lr': lr,
        },
    ]
  else:
    groups = [{'params': model.parameters(), 'lr': 1e-4}]

  opt = torch.optim.AdamW(groups, weight_decay=1e-5)
  sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
      opt, mode='max', factor=0.5, patience=3
  )

  best, best_state, bad = -1, None, 0
  for ep in range(epochs):
    model.train()
    if isinstance(model, SpikeECG):
      for p in model.backbone.parameters():
        p.requires_grad = ep >= 3
    tl, nb = 0.0, 0
    for x, y in ltr:
      x, y = x.to(device), y.to(device)
      out = model(x)
      loss = F.cross_entropy(out, y, label_smoothing=0.05)
      if spiking and model_id != 'S7':
        loss = loss + 0.01 * model.rate_loss()
      opt.zero_grad()
      loss.backward()
      nn.utils.clip_grad_norm_(model.parameters(), 2.0)
      opt.step()
      tl += loss.item()
      nb += 1

    p, y = evaluate_loader(model, lva, device)
    vf1 = f1_score(
        y, p.argmax(1), average='macro', labels=range(n_cls), zero_division=0
    )
    sched.step(vf1)
    if vf1 > best:
      best, bad, best_state = vf1, 0, copy.deepcopy(model.state_dict())
    else:
      bad += 1
      if bad >= 10:
        break

  model.load_state_dict(best_state)
  torch.save(best_state, os.path.join(out_dir, f'{model_id}_s{seed}.pt'))
  p, y = evaluate_loader(model, lte, device)

  res = dict(
      model=model_id,
      seed=seed,
      best_val_f1=float(best),
      params=int(sum(q.numel() for q in model.parameters())),
      test=compute_metrics(y, p.argmax(1), n_cls),
  )
  np.savez(
      os.path.join(out_dir, f'{model_id}_s{seed}_preds.npz'), probs=p, y=y
  )

  # 45nm CMOS Neuromorphic Energy Accounting
  if spiking:
    neurons = model.neurons()
    sums = np.zeros(len(neurons))
    n_img = 0
    with torch.no_grad():
      for bx, _ in lte:
        model(bx.to(device))
        n_img += bx.shape[0]
        for ni, ne in enumerate(neurons):
          sums[ni] += ne.count
    spk = float(sums.sum() / max(n_img, 1))
    sops = spk * 64.0
    mac_bb = 542528.0 * 12
    mac_hd = 146240.0
    res['energy'] = dict(
        spikes=spk,
        sops=sops,
        e_snn_uJ=(E_MAC * mac_bb + E_AC * sops) * 1e6,
        e_ann_uJ=(E_MAC * (mac_bb + mac_hd)) * 1e6,
    )
    if model.fusion.mode == 'lead_aware':
      res['lead_gates'] = torch.sigmoid(model.fusion.gam).tolist()

  json.dump(
      res, open(os.path.join(out_dir, f'{model_id}_s{seed}.json'), 'w'), indent=2
  )
  print(
      f"[{model_id} Seed {seed}] Accuracy: {res['test']['acc']*100:.2f}% |"
      f" Macro-F1: {res['test']['macro_f1']:.4f}"
  )
  return res


print('✓ Engine ready.')

✓ Engine ready.


## Part A - details of the runs you already have (optional)
First upload your earlier files (`S1_s0.pt`, `A4_s0.pt`, `A1_s0.pt`, `A2r_s0.pt` ... and `_preds.npz` if you have them) into the folder `/content/work/results` using the Files panel on the left. If you do not have them, skip this cell.


In [ ]:
# Cell 12: EXTRA A - per-seed results, confusion matrices, McNemar vs A1 / ResNet-50, learned lead gates
# (needs the earlier runs: files like S1_s0.pt in /content/work/results. If that folder is empty,
#  upload your saved .pt files there first, or skip this cell.)
import numpy as np, json, os
from scipy.stats import binomtest
from sklearn.metrics import f1_score, confusion_matrix

CL = list(classes); NC = len(CL)
PAGE = ['I','aVR','V1','V4','II','aVL','V2','V5','III','aVF','V3','V6']
meta = json.load(open(os.path.join(CACHE_DIR, 'meta.json')))
y_te = np.array(meta['y'])[np.array(meta['test'])]
MODELS = ['S1','A4','A1','A2r','S2','S3','S4','S5','S6a','S7']
SEEDS = [0, 1, 2]

def get_pred(mid, sd):
    p = os.path.join(RESULTS_DIR, f'{mid}_s{sd}_preds.npz')
    if os.path.exists(p):
        return np.load(p)['probs'].argmax(1)
    f = os.path.join(RESULTS_DIR, f'{mid}_s{sd}.pt')
    if not os.path.exists(f): return None
    m, _ = build_model(mid, NC, 12)
    m.load_state_dict(torch.load(f, map_location=device)); m.to(device)
    ld = DataLoader(ECGDataset(CACHE_DIR, meta['test'], train=False), batch_size=16, shuffle=False)
    pr, _ = evaluate_loader(m, ld, device)
    return pr.argmax(1)

P = {m: {s: get_pred(m, s) for s in SEEDS} for m in MODELS}
P = {m: {s: v for s, v in d.items() if v is not None} for m, d in P.items()}
P = {m: d for m, d in P.items() if d}
out = []
out.append("### Per-seed test results (71 test images)")
out.append("| Model | Seed 0 acc | Seed 1 acc | Seed 2 acc | Mean acc +- SD | Mean macro-F1 |")
out.append("|---|:---:|:---:|:---:|:---:|:---:|")
for m, d in P.items():
    accs = {s: 100*float((d[s] == y_te).mean()) for s in d}
    f1s = [f1_score(y_te, d[s], average='macro', labels=range(NC), zero_division=0) for s in d]
    a = np.array(list(accs.values()))
    cells = [f"{accs[s]:.2f}" if s in accs else "-" for s in SEEDS]
    out.append(f"| {m} | {cells[0]} | {cells[1]} | {cells[2]} | {a.mean():.2f} +- {(a.std(ddof=1) if len(a)>1 else 0):.2f} | {np.mean(f1s):.4f} |")

def mcnemar(a, b, y):
    ca, cb = (a == y), (b == y)
    n10 = int((ca & ~cb).sum()); n01 = int((~ca & cb).sum())
    p = 1.0 if n10 + n01 == 0 else binomtest(min(n10, n01), n10 + n01, 0.5).pvalue
    return n10, n01, p

out.append("\n### Paired McNemar tests per seed (first model right and second wrong / first wrong and second right / exact p)")
out.append("| Pair | Seed 0 | Seed 1 | Seed 2 |")
out.append("|---|:---:|:---:|:---:|")
for a, b in [('S1','A4'), ('S1','A1'), ('S1','A2r'), ('A4','A2r')]:
    if a in P and b in P:
        cells = []
        for s in SEEDS:
            if s in P[a] and s in P[b]:
                n10, n01, p = mcnemar(P[a][s], P[b][s], y_te); cells.append(f"{n10} / {n01} / p={p:.3f}")
            else: cells.append("-")
        out.append(f"| {a} vs {b} | {cells[0]} | {cells[1]} | {cells[2]} |")

if 'S1' in P:
    cm_sum = sum(confusion_matrix(y_te, P['S1'][s], labels=range(NC)) for s in P['S1'])
    out.append("\n### Confusion matrix of S1 (sum over the seeds; rows = true class, columns = predicted)")
    out.append("| True \\ Pred | " + " | ".join(CL) + " |"); out.append("|---|" + ":---:|"*NC)
    for i, c in enumerate(CL): out.append(f"| {c} | " + " | ".join(str(int(v)) for v in cm_sum[i]) + " |")

gates = []
for s in SEEDS:
    f = os.path.join(RESULTS_DIR, f'S1_s{s}.pt')
    if os.path.exists(f):
        st = torch.load(f, map_location='cpu')
        if 'fusion.gam' in st: gates.append(torch.sigmoid(st['fusion.gam']).numpy())
if gates:
    g = np.mean(gates, 0)
    out.append("\n### Learned lead gates g_k = sigmoid(gamma_k) of S1 (mean over seeds)")
    out.append("| Lead | " + " | ".join(PAGE) + " |"); out.append("|---|" + ":---:|"*12)
    out.append("| g_k | " + " | ".join(f"{v:.3f}" for v in g) + " |")
txt = "\n".join(out); print(txt)
open(os.path.join(WORK_DIR, 'report_extra_A.md'), 'w').write(txt)

## Part B - 5-fold cross-validation on all 491 images
Cell 13 trains 4 models x 5 folds = 20 runs. Cell 14 prints the tables for the paper.


In [ ]:
# Cell 13: EXTRA B - 5-fold cross-validation on ALL 491 distinct images (4 models)
# Every image is used once as a test image, so there are 491 test predictions instead of 71.
# It is resumable: if Colab stops, run the cell again and finished runs are skipped.
import os, json, numpy as np
from sklearn.model_selection import StratifiedKFold

MODELS_CV = ['S1', 'A4', 'A1', 'A2r']     # proposed SNN, matched ANN, LW-DN161-like, ResNet-50
N_FOLDS, REPEATS = 5, 1                   # 5x1 first; change REPEATS to 2 for 5x2 if time allows
CV_DIR = os.path.join(WORK_DIR, 'cv'); os.makedirs(CV_DIR, exist_ok=True)
meta_full = json.load(open(os.path.join(CACHE_DIR, 'meta.json')))
y_all = np.array(meta_full['y']); n_all = len(y_all)

def make_fold(rep, k):
    d = os.path.join(CV_DIR, f'r{rep}_f{k}'); os.makedirs(d, exist_ok=True)
    mp = os.path.join(d, 'meta.json')
    if not os.path.exists(mp):
        tv, te = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=100 + rep).split(np.zeros(n_all), y_all))[k]
        a, b = list(StratifiedKFold(6, shuffle=True, random_state=42).split(np.zeros(len(tv)), y_all[tv]))[0]
        m = dict(meta_full); m.update(train=tv[a].tolist(), val=tv[b].tolist(), test=te.tolist())
        json.dump(m, open(mp, 'w'))
    xp = os.path.join(d, 'X.npy')
    if not os.path.exists(xp): os.symlink(os.path.join(CACHE_DIR, 'X.npy'), xp)
    return d

for rep in range(REPEATS):
    for k in range(N_FOLDS):
        d = make_fold(rep, k); out = os.path.join(d, 'results'); os.makedirs(out, exist_ok=True)
        sd = k + 10 * rep
        for mid in MODELS_CV:
            if os.path.exists(os.path.join(out, f'{mid}_s{sd}.json')):
                print('skip', mid, 'repeat', rep, 'fold', k); continue
            print(f'--- {mid} | repeat {rep} | fold {k} ---')
            execute_train_run(mid, d, out, seed=sd, epochs=35)
            pt = os.path.join(out, f'{mid}_s{sd}.pt')
            if os.path.exists(pt): os.remove(pt)      # keep the disk free
print('All cross-validation runs finished.')

--- S1 | repeat 0 | fold 0 ---
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 122MB/s]


[S1 Seed 0] Accuracy: 88.89% | Macro-F1: 0.8680
--- A4 | repeat 0 | fold 0 ---
[A4 Seed 0] Accuracy: 91.92% | Macro-F1: 0.8821
--- A1 | repeat 0 | fold 0 ---
Downloading: "https://download.pytorch.org/models/densenet161-8d451a50.pth" to /root/.cache/torch/hub/checkpoints/densenet161-8d451a50.pth


100%|██████████| 110M/110M [00:01<00:00, 109MB/s]


[A1 Seed 0] Accuracy: 90.91% | Macro-F1: 0.8785
--- A2r | repeat 0 | fold 0 ---
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s]


[A2r Seed 0] Accuracy: 92.93% | Macro-F1: 0.8990
--- S1 | repeat 0 | fold 1 ---
[S1 Seed 1] Accuracy: 89.80% | Macro-F1: 0.7475
--- A4 | repeat 0 | fold 1 ---
[A4 Seed 1] Accuracy: 90.82% | Macro-F1: 0.8330
--- A1 | repeat 0 | fold 1 ---
[A1 Seed 1] Accuracy: 88.78% | Macro-F1: 0.8190
--- A2r | repeat 0 | fold 1 ---
[A2r Seed 1] Accuracy: 94.90% | Macro-F1: 0.8895
--- S1 | repeat 0 | fold 2 ---
[S1 Seed 2] Accuracy: 83.67% | Macro-F1: 0.7512
--- A4 | repeat 0 | fold 2 ---
[A4 Seed 2] Accuracy: 81.63% | Macro-F1: 0.7713
--- A1 | repeat 0 | fold 2 ---
[A1 Seed 2] Accuracy: 89.80% | Macro-F1: 0.8594
--- A2r | repeat 0 | fold 2 ---
[A2r Seed 2] Accuracy: 93.88% | Macro-F1: 0.9469
--- S1 | repeat 0 | fold 3 ---
[S1 Seed 3] Accuracy: 85.71% | Macro-F1: 0.7572
--- A4 | repeat 0 | fold 3 ---
[A4 Seed 3] Accuracy: 90.82% | Macro-F1: 0.8677
--- A1 | repeat 0 | fold 3 ---
[A1 Seed 3] Accuracy: 92.86% | Macro-F1: 0.9219
--- A2r | repeat 0 | fold 3 ---
[A2r Seed 3] Accuracy: 92.86% | Macro-F1: 0.89

In [ ]:
# Cell 14: EXTRA B report - cross-validation tables for the paper
import os, json, numpy as np
from scipy.stats import binomtest, ttest_rel, wilcoxon
from sklearn.metrics import confusion_matrix, f1_score, cohen_kappa_score

CL = list(classes); NC = len(CL)
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = p + z*z/(2*n); a = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return 100*(c-a)/d, 100*(c+a)/d

OOF = {m: {} for m in MODELS_CV}; FOLD_ACC = {m: [] for m in MODELS_CV}; EN = []
for rep in range(REPEATS):
    for m in MODELS_CV: OOF[m][rep] = np.full(n_all, -1)
    for k in range(N_FOLDS):
        d = os.path.join(CV_DIR, f'r{rep}_f{k}'); sd = k + 10*rep
        te = np.array(json.load(open(os.path.join(d, 'meta.json')))['test'])
        for m in MODELS_CV:
            p = os.path.join(d, 'results', f'{m}_s{sd}_preds.npz')
            if not os.path.exists(p): continue
            pred = np.load(p)['probs'].argmax(1)
            OOF[m][rep][te] = pred
            FOLD_ACC[m].append(100*float((pred == y_all[te]).mean()))
            if m == 'S1':
                j = json.load(open(os.path.join(d, 'results', f'{m}_s{sd}.json')))
                if 'energy' in j: EN.append(j['energy'])

done = [m for m in MODELS_CV if all((OOF[m][r] >= 0).all() for r in range(REPEATS))]
out = [f"### {N_FOLDS}-fold cross-validation x {REPEATS} on all {n_all} distinct images"]
out.append("| Model | Fold accuracy (mean +- SD) | Pooled accuracy (all %d images) | 95%% Wilson interval | Macro-F1 | Kappa |" % n_all)
out.append("|---|:---:|:---:|:---:|:---:|:---:|")
for m in done:
    fa = np.array(FOLD_ACC[m])
    accs = [100*float((OOF[m][r] == y_all).mean()) for r in range(REPEATS)]
    k0 = int((OOF[m][0] == y_all).sum()); lo, hi = wilson(k0, n_all)
    f1 = np.mean([f1_score(y_all, OOF[m][r], average='macro') for r in range(REPEATS)])
    kp = np.mean([cohen_kappa_score(y_all, OOF[m][r]) for r in range(REPEATS)])
    out.append(f"| {m} | {fa.mean():.2f} +- {fa.std(ddof=1):.2f} | {np.mean(accs):.2f} | {lo:.1f}-{hi:.1f} | {f1:.4f} | {kp:.3f} |")

if 'S1' in done:
    out.append("\n### Paired comparison of S1 with the other models (pooled predictions of repeat 0, McNemar exact test; fold accuracies, paired t-test and Wilcoxon)")
    out.append("| S1 vs | S1 right / other wrong | S1 wrong / other right | McNemar p | paired t-test p | Wilcoxon p |")
    out.append("|---|:---:|:---:|:---:|:---:|:---:|")
    for m in [x for x in done if x != 'S1']:
        a, b = OOF['S1'][0] == y_all, OOF[m][0] == y_all
        n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
        pm = 1.0 if n10 + n01 == 0 else binomtest(min(n10, n01), n10 + n01, 0.5).pvalue
        fa, fb = np.array(FOLD_ACC['S1']), np.array(FOLD_ACC[m])
        pt = ttest_rel(fa, fb).pvalue
        try: pw = wilcoxon(fa, fb).pvalue
        except Exception: pw = float('nan')
        out.append(f"| {m} | {n10} | {n01} | {pm:.3f} | {pt:.3f} | {pw:.3f} |")
    cm = confusion_matrix(y_all, OOF['S1'][0], labels=range(NC))
    out.append("\n### S1: confusion matrix on all images (rows = true class, columns = predicted)")
    out.append("| True \\ Pred | " + " | ".join(CL) + " |"); out.append("|---|" + ":---:|"*NC)
    for i, c in enumerate(CL): out.append(f"| {c} | " + " | ".join(str(int(v)) for v in cm[i]) + " |")
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
    P_ = tp/np.maximum(tp+fp, 1); R_ = tp/np.maximum(tp+fn, 1); S_ = tn/np.maximum(tn+fp, 1); F_ = 2*P_*R_/np.maximum(P_+R_, 1e-12)
    out.append("\n### S1: per-class results on all images")
    out.append("| Class | Precision | Sensitivity | Specificity | F1 | Support |"); out.append("|---|:---:|:---:|:---:|:---:|:---:|")
    for i, c in enumerate(CL): out.append(f"| {c} | {P_[i]:.3f} | {R_[i]:.3f} | {S_[i]:.3f} | {F_[i]:.3f} | {int(cm[i].sum())} |")

if EN:
    E_MAC_, E_AC_ = 4.6e-12, 0.9e-12
    snn = np.mean([e['e_snn_uJ'] for e in EN]); ann = np.mean([e['e_ann_uJ'] for e in EN])
    h_snn = np.mean([E_AC_*e['sops']*1e6 for e in EN]); h_ann = E_MAC_*146240.0*1e6
    out.append("\n### Estimated energy (45 nm model, mean over folds)")
    out.append(f"- S1 whole pipeline {snn:.2f} uJ, matched ANN {ann:.2f} uJ, ratio {snn/ann:.3f}")
    out.append(f"- Spiking head {h_snn:.2f} uJ against {h_ann:.2f} uJ for the matched ANN head, ratio {h_snn/h_ann:.3f}")
txt = "\n".join(out); print(txt)
open(os.path.join(WORK_DIR, 'report_cv.md'), 'w').write(txt)

### 5-fold cross-validation x 1 on all 491 distinct images
| Model | Fold accuracy (mean +- SD) | Pooled accuracy (all 491 images) | 95% Wilson interval | Macro-F1 | Kappa |
|---|:---:|:---:|:---:|:---:|:---:|
| S1 | 86.15 +- 3.14 | 86.15 | 82.8-88.9 | 0.7660 | 0.789 |
| A4 | 89.00 +- 4.18 | 89.00 | 85.9-91.5 | 0.8389 | 0.834 |
| A1 | 90.63 +- 1.52 | 90.63 | 87.7-92.9 | 0.8711 | 0.857 |
| A2r | 92.87 +- 1.91 | 92.87 | 90.2-94.8 | 0.9037 | 0.892 |

### Paired comparison of S1 with the other models (pooled predictions of repeat 0, McNemar exact test; fold accuracies, paired t-test and Wilcoxon)
| S1 vs | S1 right / other wrong | S1 wrong / other right | McNemar p | paired t-test p | Wilcoxon p |
|---|:---:|:---:|:---:|:---:|:---:|
| A4 | 14 | 28 | 0.044 | 0.148 | 0.188 |
| A1 | 16 | 38 | 0.004 | 0.060 | 0.125 |
| A2r | 16 | 49 | 0.000 | 0.003 | 0.062 |

### S1: confusion matrix on all images (rows = true class, columns = predicted)
| True \ Pred | Abnormal Heartbeat | History of MI | Myo

1772

## Part C - add rendered PTB-XL images to the training set
Cell 15 draws the PTB-XL records as images, Cell 16 trains S1, A4 and A1 (3 seeds each), Cell 17 compares with Kaggle-only training. The PTB-XL test records (fold 10) are never used for training.


In [ ]:
# Cell 15: EXTRA C (part 1) - draw PTB-XL records as images: training images (folds 1-8) and the test images (fold 10)
!pip -q install wfdb kagglehub
import os, ast, json, glob, numpy as np, pandas as pd, wfdb, kagglehub
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

root = kagglehub.dataset_download('khyeh0719/ptb-xl-dataset')
PTB_DIR = os.path.dirname(glob.glob(os.path.join(root, '**', 'ptbxl_database.csv'), recursive=True)[0])
print('PTB-XL at', PTB_DIR)
db  = pd.read_csv(os.path.join(PTB_DIR, 'ptbxl_database.csv'), index_col='ecg_id')
scp = pd.read_csv(os.path.join(PTB_DIR, 'scp_statements.csv'), index_col=0); scp = scp[scp.diagnostic == 1]
def super_classes(d):
    codes = ast.literal_eval(d)
    return {scp.loc[k, 'diagnostic_class'] for k, v in codes.items() if k in scp.index and v >= 50}
db['sup'] = db.scp_codes.apply(super_classes)

def pick(folds, n_per, seed):
    sub = db[db.strat_fold.isin(folds)].drop_duplicates('patient_id')
    norm = sub[sub.sup.apply(lambda s: s == {'NORM'})]; mi = sub[sub.sup.apply(lambda s: s == {'MI'})]
    n = min(n_per, len(norm), len(mi))
    return pd.concat([norm.sample(n, random_state=seed).assign(lab='NORM'), mi.sample(n, random_state=seed).assign(lab='MI')])

sel_test = pick([10], 150, 0)                 # the same 150 + 150 records as before
sel_train = pick(list(range(1, 9)), 250, 1)   # new training records, folds 1-8 only
print('test', len(sel_test), 'train', len(sel_train))

H, W = 96, 128
PAGE      = ['I','aVR','V1','V4','II','aVL','V2','V5','III','aVF','V3','V6']
PTB_ORDER = ['I','II','III','AVR','AVL','AVF','V1','V2','V3','V4','V5','V6']
col = {n: PTB_ORDER.index(n.upper()) for n in PAGE}
fig = plt.figure(figsize=(W/100, H/100), dpi=100)
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 2.5); ax.set_ylim(-1.6, 1.6)
ax.set_xticks(np.arange(0, 2.5, 0.2)); ax.set_yticks(np.arange(-1.5, 1.6, 0.5))
ax.set_xticks(np.arange(0, 2.5, 0.04), minor=True); ax.set_yticks(np.arange(-1.6, 1.7, 0.1), minor=True)
ax.grid(which='minor', color='#f6d5d5', lw=0.3); ax.grid(which='major', color='#eba5a5', lw=0.6)
ax.tick_params(which='both', length=0, labelbottom=False, labelleft=False); ax.set_facecolor('#fff7f7')
line, = ax.plot([], [], color='#2b2b2b', lw=0.9); lab = ax.text(0.05, 1.25, '', fontsize=5)
def render(sig, name):
    s = sig[30:280] - np.median(sig[30:280])
    line.set_data(np.arange(len(s)) / 100.0, s); lab.set_text(name); fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]; assert img.shape[:2] == (H, W), img.shape
    return img.copy()
def render_all(sel):
    X = np.zeros((len(sel), 12, 3, H, W), np.uint8)
    for i, fn in enumerate(sel.filename_lr):
        sig, _ = wfdb.rdsamp(os.path.join(PTB_DIR, 'records100', fn.replace('records100/', '')))
        for k, n in enumerate(PAGE): X[i, k] = render(sig[:, col[n]], n).transpose(2, 0, 1)
        if (i + 1) % 100 == 0: print('  drew', i + 1, '/', len(sel))
    return X, (sel.lab.values == 'MI').astype(int)
X_ptb_test, y_ptb_test = render_all(sel_test)
X_ptb_train, y_ptb_train = render_all(sel_train)
np.save(os.path.join(WORK_DIR, 'X_ptb_train.npy'), X_ptb_train); np.save(os.path.join(WORK_DIR, 'y_ptb_train.npy'), y_ptb_train)
np.save(os.path.join(WORK_DIR, 'X_ptb_test.npy'), X_ptb_test);  np.save(os.path.join(WORK_DIR, 'y_ptb_test.npy'), y_ptb_test)
print('rendered', X_ptb_train.shape, X_ptb_test.shape)

Using Colab cache for faster access to the 'ptb-xl-dataset' dataset.
PTB-XL at /kaggle/input/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1
test 300 train 500
  drew 100 / 300
  drew 200 / 300
  drew 300 / 300
  drew 100 / 500
  drew 200 / 500
  drew 300 / 500
  drew 400 / 500
  drew 500 / 500
rendered (500, 12, 3, 96, 128) (300, 12, 3, 96, 128)


In [ ]:
# Cell 16: EXTRA C (part 2) - train with Kaggle images + rendered PTB-XL images (Normal and MI), then test on both
import os, json, numpy as np, torch
CL = list(classes)
Xk = np.load(os.path.join(CACHE_DIR, 'X.npy'), mmap_mode='r')
meta_k = json.load(open(os.path.join(CACHE_DIR, 'meta.json')))
MIX_DIR = os.path.join(WORK_DIR, 'mixed'); MIX_RES = os.path.join(MIX_DIR, 'results')
os.makedirs(MIX_RES, exist_ok=True)

n_k, n_extra = len(Xk), len(X_ptb_train)
if not os.path.exists(os.path.join(MIX_DIR, 'X.npy')):
    np.save(os.path.join(MIX_DIR, 'X.npy'), np.concatenate([np.array(Xk), X_ptb_train]))
y_extra = [CL.index('Myocardial Infarction') if v == 1 else CL.index('Normal') for v in y_ptb_train]
m = dict(meta_k); m['y'] = list(meta_k['y']) + y_extra
m['train'] = list(meta_k['train']) + list(range(n_k, n_k + n_extra))     # val and test stay Kaggle-only
json.dump(m, open(os.path.join(MIX_DIR, 'meta.json'), 'w'))
print('training images:', len(m['train']), '(Kaggle', len(meta_k['train']), '+ PTB-XL', n_extra, ')')

for mid in ['S1', 'A4', 'A1']:
    for sd in [0, 1, 2]:
        if os.path.exists(os.path.join(MIX_RES, f'{mid}_s{sd}.json')): print('skip', mid, sd); continue
        print(f'--- mixed training {mid} seed {sd} ---')
        execute_train_run(mid, MIX_DIR, MIX_RES, seed=sd, epochs=35)
print('Mixed training finished.')

training images: 850 (Kaggle 350 + PTB-XL 500 )
--- mixed training S1 seed 0 ---
[S1 Seed 0] Accuracy: 94.37% | Macro-F1: 0.8589
--- mixed training S1 seed 1 ---
[S1 Seed 1] Accuracy: 91.55% | Macro-F1: 0.8363
--- mixed training S1 seed 2 ---
[S1 Seed 2] Accuracy: 91.55% | Macro-F1: 0.8743
--- mixed training A4 seed 0 ---
[A4 Seed 0] Accuracy: 95.77% | Macro-F1: 0.9402
--- mixed training A4 seed 1 ---
[A4 Seed 1] Accuracy: 92.96% | Macro-F1: 0.8867
--- mixed training A4 seed 2 ---
[A4 Seed 2] Accuracy: 94.37% | Macro-F1: 0.9271
--- mixed training A1 seed 0 ---
[A1 Seed 0] Accuracy: 90.14% | Macro-F1: 0.8526
--- mixed training A1 seed 1 ---
[A1 Seed 1] Accuracy: 94.37% | Macro-F1: 0.9039
--- mixed training A1 seed 2 ---
[A1 Seed 2] Accuracy: 91.55% | Macro-F1: 0.8806
Mixed training finished.


In [ ]:
# Cell 17: EXTRA C (part 3) - compare Kaggle-only training with Kaggle + PTB-XL training
import os, json, numpy as np, torch
CL = list(classes); MI_IDX = [CL.index('Myocardial Infarction'), CL.index('History of MI')]
@torch.no_grad()
def predict(model, X):
    model.eval(); o = []
    for i in range(0, len(X), 16):
        x = torch.from_numpy(X[i:i+16]).float() / 255.0
        o.append(model(((x - MEAN) / STD).to(device)).argmax(-1).cpu())
    return torch.cat(o).numpy()
def binary_stats(pred4, y_mi):
    p = np.isin(pred4, MI_IDX).astype(int)
    tp = ((p == 1) & (y_mi == 1)).sum(); tn = ((p == 0) & (y_mi == 0)).sum()
    se = tp / max((y_mi == 1).sum(), 1); sp = tn / max((y_mi == 0).sum(), 1)
    return dict(acc=100*(tp+tn)/len(p), sens=100*se, spec=100*sp, bal=50*(se+sp))
def ms(rs, k):
    v = np.array([r[k] for r in rs]); return f"{v.mean():.1f} +- {(v.std(ddof=1) if len(v) > 1 else 0):.1f}"

meta = json.load(open(os.path.join(CACHE_DIR, 'meta.json')))
Xc = np.load(os.path.join(CACHE_DIR, 'X.npy'), mmap_mode='r')
te = np.array(meta['test']); yk = np.array(meta['y'])[te]
keep = np.isin(yk, [CL.index('Normal')] + MI_IDX)
X_k = np.array(Xc[te[keep]]); y_k = np.isin(yk[keep], MI_IDX).astype(int)

rows = []
for tag, folder in [('Kaggle only', RESULTS_DIR), ('Kaggle + PTB-XL', MIX_RES)]:
    for mid in ['S1', 'A4', 'A1']:
        rp, rk, r4 = [], [], []
        for sd in [0, 1, 2]:
            f = os.path.join(folder, f'{mid}_s{sd}.pt')
            if not os.path.exists(f): continue
            mdl, _ = build_model(mid, len(CL), 12)
            mdl.load_state_dict(torch.load(f, map_location=device)); mdl.to(device)
            rp.append(binary_stats(predict(mdl, X_ptb_test), y_ptb_test))
            rk.append(binary_stats(predict(mdl, X_k), y_k))
            jf = os.path.join(folder, f'{mid}_s{sd}.json')
            if os.path.exists(jf): r4.append(dict(acc=100*json.load(open(jf))['test']['acc']))
        if rp:
            a4 = ms(r4, 'acc') if r4 else '-'
            rows.append(f"| {tag} | {mid} | {len(rp)} | {a4} | {ms(rk,'acc')} | {ms(rp,'acc')} | {ms(rp,'sens')} | {ms(rp,'spec')} | {ms(rp,'bal')} |")
md = ["### Effect of adding rendered PTB-XL images to the training set (test: held-out Kaggle images and PTB-XL fold 10)",
      f"PTB-XL test images: {len(y_ptb_test)} ({int((y_ptb_test==0).sum())} Normal, {int((y_ptb_test==1).sum())} MI); PTB-XL training images added: {len(X_ptb_train)}",
      "| Training data | Model | Seeds | Kaggle 4-class acc (%) | Kaggle Normal/MI acc (%) | PTB-XL acc (%) | PTB-XL MI sensitivity (%) | PTB-XL Normal specificity (%) | PTB-XL balanced acc (%) |",
      "|---|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|"] + rows
txt = "\n".join(md); print(txt)
open(os.path.join(WORK_DIR, 'report_mixed.md'), 'w').write(txt)

### Effect of adding rendered PTB-XL images to the training set (test: held-out Kaggle images and PTB-XL fold 10)
PTB-XL test images: 300 (150 Normal, 150 MI); PTB-XL training images added: 500
| Training data | Model | Seeds | Kaggle 4-class acc (%) | Kaggle Normal/MI acc (%) | PTB-XL acc (%) | PTB-XL MI sensitivity (%) | PTB-XL Normal specificity (%) | PTB-XL balanced acc (%) |
|---|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Kaggle + PTB-XL | S1 | 3 | 94.4 +- 2.8 | 93.9 +- 1.5 | 82.4 +- 1.1 | 76.4 +- 0.4 | 88.4 +- 1.9 | 82.4 +- 1.1 |
| Kaggle + PTB-XL | A4 | 3 | 92.0 +- 2.2 | 93.0 +- 3.0 | 81.8 +- 2.7 | 75.6 +- 12.6 | 88.0 +- 7.2 | 81.8 +- 2.7 |
| Kaggle + PTB-XL | A1 | 3 | 92.0 +- 1.6 | 91.2 +- 1.5 | 84.4 +- 1.3 | 84.7 +- 5.0 | 84.2 +- 5.0 | 84.4 +- 1.3 |


774

## Save the results


In [ ]:
# Cell 18: pack the small result files for download (no model weights)
import os, zipfile
from google.colab import files
zp = '/content/extra_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for base, _, fs in os.walk(WORK_DIR):
        for f in fs:
            if f.endswith(('.md', '.json', '.npz', '.png', '.csv')) and 'cache' not in base:
                z.write(os.path.join(base, f), os.path.relpath(os.path.join(base, f), WORK_DIR))
print('zip size (MB):', round(os.path.getsize(zp) / 1e6, 1))
files.download(zp)

zip size (MB): 0.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 19: PLUS step 0 - check that everything from the earlier cells is still there (and restore if Colab was reset)
import os, json, glob
print('WORK_DIR:', WORK_DIR, '| CACHE_DIR:', CACHE_DIR)
need = [os.path.join(WORK_DIR, 'cv', f'r0_f{k}', 'results', f'{m}_s{k}_preds.npz') for k in range(5) for m in ['S1', 'A4', 'A1', 'A2r']]
missing = [p for p in need if not os.path.exists(p)]
print('baseline cross-validation prediction files found:', len(need) - len(missing), 'of', len(need))
if missing:
    print('Some baseline files are missing (Colab was reset?). Upload extra_results.zip (the one you downloaded from Cell 18, the file that has the cv folder).')
    from google.colab import files
    import zipfile
    up = files.upload()
    for name in up:
        if name.endswith('.zip'):
            zipfile.ZipFile(name).extractall(WORK_DIR)
    missing = [p for p in need if not os.path.exists(p)]
    print('still missing:', len(missing))
assert not missing, 'Baseline CV results not found: the comparison needs them.'
assert os.path.exists(os.path.join(CACHE_DIR, 'X.npy')), 'cache X.npy not found: run Cells 1, 2 (data set-up) again'
for nm in ['build_model', 'execute_train_run', 'ECGDataset', 'compute_metrics', 'evaluate_loader', 'SpikeECG', 'Fusion', 'LIF']:
    assert nm in globals(), f'{nm} is missing: run Cells 1, 2, 4, 5 first'
print('OK - ready for Cell 20')

WORK_DIR: /content/work | CACHE_DIR: /content/work/cache
baseline cross-validation prediction files found: 0 of 20
Some baseline files are missing (Colab was reset?). Upload extra_results.zip (the one you downloaded from Cell 18, the file that has the cv folder).


Saving extra_results_cv.zip to extra_results_cv.zip
still missing: 0
OK - ready for Cell 20


In [ ]:
# Cell 20: PLUS step 1 - definitions: S1+ (membrane read-out), focal loss, distillation, training function
import os, json, copy, random, math
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score

class FusionMem(Fusion):
    """Same lead-aware fusion, but the last layer is NOT spiking: a leaky integrator (membrane potential) is averaged over time."""
    def __init__(self, in_dim, K, n_cls):
        super().__init__(in_dim, K, n_cls, 'lead_aware', ann=False)
        self.mem_logit = nn.Parameter(torch.tensor(0.0))       # leak of the integrator, sigmoid(0) = 0.5
    def forward(self, c):
        T, B = c.shape[:2]
        g = torch.sigmoid(self.gam); gs = []
        for fc, neu, idx in zip(self.gfc, self.gneu, self.groups):
            gs.append(neu(fc((c[:, :, idx, :] * g[idx][None, None, :, None]).sum(dim=2))))
        h = self.fus(torch.cat(gs, -1))                        # [T, B, 128] input current
        b = torch.sigmoid(self.mem_logit); v = torch.zeros_like(h[0]); acc = 0
        for t in range(T):
            v = b * v + h[t]; acc = acc + v
        return self.out(acc / T)

class SpikeECGPlus(SpikeECG):
    def __init__(self, n_cls, K=12, readout='spike'):
        super().__init__(n_cls, K, 'polar_temporal', 'lead_aware', 8, ann=False)
        if readout == 'membrane':
            self.fusion = FusionMem(self.head.out_dim, K, n_cls)

# name: (readout, focal gamma, distillation alpha, rate-loss weight)
CONFIGS = {
    'P1_focal':        dict(readout='spike',    gamma=2.0, kd=0.0, lam=0.01),
    'P2_membrane':     dict(readout='membrane', gamma=0.0, kd=0.0, lam=0.01),
    'P3_distill':      dict(readout='spike',    gamma=0.0, kd=0.5, lam=0.01),
    'P4_distill_focal':dict(readout='spike',    gamma=2.0, kd=0.5, lam=0.01),
    'P5_distill_mem':  dict(readout='membrane', gamma=0.0, kd=0.5, lam=0.01),
    'P6_full':         dict(readout='membrane', gamma=2.0, kd=0.5, lam=0.01),   # distillation + focal loss + membrane read-out
    'P7_full_lowrate': dict(readout='membrane', gamma=2.0, kd=0.5, lam=0.003),
}
KD_TEMP = 2.0

def train_plus(name, cfg, cache_dir, out_dir, seed, teacher=None, epochs=35, bs=16, lr=1e-3):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    meta = json.load(open(os.path.join(cache_dir, 'meta.json'))); n_cls = len(meta['classes'])
    tr = ECGDataset(cache_dir, meta['train'], train=True, seed=seed)
    va = ECGDataset(cache_dir, meta['val']); te = ECGDataset(cache_dir, meta['test'])
    cnt = np.bincount(tr.y, minlength=n_cls); w = (1.0 / np.maximum(cnt, 1))[tr.y]
    sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(tr), replacement=True)
    ltr = DataLoader(tr, batch_size=bs, sampler=sampler, num_workers=2, pin_memory=True)
    lva = DataLoader(va, batch_size=bs, shuffle=False, num_workers=2)
    lte = DataLoader(te, batch_size=bs, shuffle=False, num_workers=2)
    model = SpikeECGPlus(n_cls, meta['K'], cfg['readout']).to(device)
    bb = list(model.backbone.parameters()); ids = {id(p) for p in bb}
    opt = torch.optim.AdamW([{'params': bb, 'lr': 1e-4},
                             {'params': [p for p in model.parameters() if id(p) not in ids], 'lr': lr}], weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for p in model.backbone.parameters(): p.requires_grad = ep >= 3
        for x, y in ltr:
            x, y = x.to(device), y.to(device)
            out = model(x)
            ce = F.cross_entropy(out, y, label_smoothing=0.05, reduction='none')
            if cfg['gamma'] > 0: ce = ((1 - torch.exp(-ce)) ** cfg['gamma']) * ce       # focal loss
            loss = ce.mean()
            if cfg['kd'] > 0 and teacher is not None:
                with torch.no_grad(): tl = teacher(x)
                kd = F.kl_div(F.log_softmax(out / KD_TEMP, -1), F.softmax(tl / KD_TEMP, -1), reduction='batchmean') * KD_TEMP ** 2
                loss = (1 - cfg['kd']) * loss + cfg['kd'] * kd
            loss = loss + cfg['lam'] * model.rate_loss()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step()
        p, yv = evaluate_loader(model, lva, device)
        vf1 = f1_score(yv, p.argmax(1), average='macro', labels=range(n_cls), zero_division=0)
        sched.step(vf1)
        if vf1 > best: best, bad, best_state = vf1, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= 10: break
    model.load_state_dict(best_state)
    p, yt = evaluate_loader(model, lte, device)
    res = dict(model=name, config=cfg, seed=seed, best_val_f1=float(best), params=int(sum(q.numel() for q in model.parameters())),
               test=compute_metrics(yt, p.argmax(1), n_cls))
    np.savez(os.path.join(out_dir, f'{name}_s{seed}_preds.npz'), probs=p, y=yt)
    neurons = model.neurons(); sums = np.zeros(len(neurons)); n_img = 0
    with torch.no_grad():
        for bx, _ in lte:
            model(bx.to(device)); n_img += bx.shape[0]
            for i, ne in enumerate(neurons): sums[i] += ne.count
    spk = float(sums.sum() / max(n_img, 1)); sops = spk * 64.0
    res['energy'] = dict(spikes=spk, sops=sops, e_snn_uJ=(E_MAC * 542528.0 * 12 + E_AC * sops) * 1e6,
                         e_ann_uJ=(E_MAC * (542528.0 * 12 + 146240.0)) * 1e6)
    res['lead_gates'] = torch.sigmoid(model.fusion.gam).tolist()
    json.dump(res, open(os.path.join(out_dir, f'{name}_s{seed}.json'), 'w'), indent=2)
    print(f"[{name} seed {seed}] val-F1 {best:.3f} | test acc {res['test']['acc']*100:.2f}% | macro-F1 {res['test']['macro_f1']:.4f}")
    return res
print('OK - definitions ready (configs:', list(CONFIGS), ')')

OK - definitions ready (configs: ['P1_focal', 'P2_membrane', 'P3_distill', 'P4_distill_focal', 'P5_distill_mem', 'P6_full', 'P7_full_lowrate'] )


In [ ]:
# Cell 20b: keep PLUS results safe on Google Drive (run once after Cell 20)
from google.colab import drive
import os, shutil, threading, time
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/ecg_snn_plus'; os.makedirs(SAVE, exist_ok=True)
CV_DIR = os.path.join(WORK_DIR, 'cv')
def restore():                                   # copy back anything already saved
    for k in range(5):
        s = os.path.join(SAVE, f'r0_f{k}', 'plus'); d = os.path.join(CV_DIR, f'r0_f{k}', 'plus')
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
def backup_loop():
    while True:
        for k in range(5):
            s = os.path.join(CV_DIR, f'r0_f{k}', 'plus')
            if os.path.isdir(s): shutil.copytree(s, os.path.join(SAVE, f'r0_f{k}', 'plus'), dirs_exist_ok=True)
        time.sleep(120)
restore(); threading.Thread(target=backup_loop, daemon=True).start()
print('Backup to Drive is running every 2 minutes. Saved so far:', os.listdir(SAVE))

Mounted at /content/drive
Backup to Drive is running every 2 minutes. Saved so far: []


In [ ]:
import torch; print(torch.cuda.is_available(), device)

True cuda


In [ ]:
# Cell 21: PLUS step 2 - run S1+ variants on the SAME 5 folds as the baseline (resumable; teacher = ResNet-50 trained inside each fold on that fold's training images only)
import os, json, time, numpy as np, torch
from sklearn.model_selection import StratifiedKFold
RUN_LIST = ['P6_full', 'P3_distill', 'P1_focal', 'P2_membrane']   # start with these four; later add 'P4_distill_focal', 'P5_distill_mem', 'P7_full_lowrate'
N_FOLDS, REPEATS = 5, 1
CV_DIR = os.path.join(WORK_DIR, 'cv')
meta_full = json.load(open(os.path.join(CACHE_DIR, 'meta.json'))); y_all = np.array(meta_full['y']); n_all = len(y_all)

def make_fold(rep, k):                       # identical to Cell 13 (same folds); only rebuilds what is missing
    d = os.path.join(CV_DIR, f'r{rep}_f{k}'); os.makedirs(d, exist_ok=True); mp = os.path.join(d, 'meta.json')
    if not os.path.exists(mp):
        tv, te = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=100 + rep).split(np.zeros(n_all), y_all))[k]
        a, b = list(StratifiedKFold(6, shuffle=True, random_state=42).split(np.zeros(len(tv)), y_all[tv]))[0]
        m = dict(meta_full); m.update(train=tv[a].tolist(), val=tv[b].tolist(), test=te.tolist()); json.dump(m, open(mp, 'w'))
    xp = os.path.join(d, 'X.npy')
    if not os.path.exists(xp): os.symlink(os.path.join(CACHE_DIR, 'X.npy'), xp)
    return d

t0 = time.time()
for rep in range(REPEATS):
    for k in range(N_FOLDS):
        d = make_fold(rep, k); sd = k + 10 * rep
        todo = [c for c in RUN_LIST if not os.path.exists(os.path.join(d, 'plus', c, f'{c}_s{sd}.json'))]
        if not todo: print('fold', k, 'all done'); continue
        teacher = None
        if any(CONFIGS[c]['kd'] > 0 for c in todo):
            tdir = os.path.join(d, 'teacher'); os.makedirs(tdir, exist_ok=True)
            tp = os.path.join(tdir, f'A2r_s{sd}.pt')
            if not os.path.exists(tp):
                print(f'--- teacher ResNet-50 | fold {k} (trained on this fold training images only) ---')
                execute_train_run('A2r', d, tdir, seed=sd, epochs=35)
            teacher, _ = build_model('A2r', len(meta_full['classes']), 12)
            teacher.load_state_dict(torch.load(tp, map_location=device)); teacher.to(device).eval()
            for q in teacher.parameters(): q.requires_grad = False
        for c in todo:
            out = os.path.join(d, 'plus', c); os.makedirs(out, exist_ok=True)
            print(f'--- {c} | fold {k} | elapsed {(time.time()-t0)/60:.0f} min ---')
            train_plus(c, CONFIGS[c], d, out, sd, teacher=teacher if CONFIGS[c]['kd'] > 0 else None)
        del teacher; torch.cuda.empty_cache()
        tp = os.path.join(d, 'teacher', f'A2r_s{sd}.pt')
        if os.path.exists(tp): os.remove(tp)                    # free disk (the teacher is retrained if the cell is re-run)
print('Finished. Total minutes:', round((time.time() - t0) / 60))

--- teacher ResNet-50 | fold 0 (trained on this fold training images only) ---
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 104MB/s]


[A2r Seed 0] Accuracy: 92.93% | Macro-F1: 0.9027
--- P6_full | fold 0 | elapsed 3 min ---
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 115MB/s]


[P6_full seed 0] val-F1 0.852 | test acc 92.93% | macro-F1 0.8718
--- P3_distill | fold 0 | elapsed 8 min ---
[P3_distill seed 0] val-F1 0.898 | test acc 93.94% | macro-F1 0.8810
--- P1_focal | fold 0 | elapsed 12 min ---
[P1_focal seed 0] val-F1 0.868 | test acc 91.92% | macro-F1 0.8761
--- P2_membrane | fold 0 | elapsed 14 min ---
[P2_membrane seed 0] val-F1 0.852 | test acc 94.95% | macro-F1 0.8907
--- teacher ResNet-50 | fold 1 (trained on this fold training images only) ---
[A2r Seed 1] Accuracy: 92.86% | Macro-F1: 0.8694
--- P6_full | fold 1 | elapsed 19 min ---
[P6_full seed 1] val-F1 0.953 | test acc 89.80% | macro-F1 0.8211
--- P3_distill | fold 1 | elapsed 23 min ---
[P3_distill seed 1] val-F1 0.945 | test acc 89.80% | macro-F1 0.8203
--- P1_focal | fold 1 | elapsed 26 min ---
[P1_focal seed 1] val-F1 0.918 | test acc 86.73% | macro-F1 0.6715
--- P2_membrane | fold 1 | elapsed 29 min ---
[P2_membrane seed 1] val-F1 0.953 | test acc 90.82% | macro-F1 0.8321
--- teacher ResNet-

In [ ]:
# Cell 22: PLUS step 3 - report: every variant vs the baseline S1 (and A4, A1, ResNet-50) on the pooled 491 test predictions
import os, json, numpy as np
from sklearn.metrics import confusion_matrix, f1_score, cohen_kappa_score
from scipy.stats import binomtest
CV_DIR = os.path.join(WORK_DIR, 'cv'); N_FOLDS = 5
meta0 = json.load(open(os.path.join(CV_DIR, 'r0_f0', 'meta.json'))); y = np.array(meta0['y']); CL = meta0['classes']; n = len(y)
MI = CL.index('Myocardial Infarction'); HMI = CL.index('History of MI')

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); a = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return 100 * (c - a) / d, 100 * (c + a) / d
def load(kind, name):
    pred = np.full(n, -1); vals = []; fa = []; eng = []
    for k in range(N_FOLDS):
        d = os.path.join(CV_DIR, f'r0_f{k}'); te = np.array(json.load(open(os.path.join(d, 'meta.json')))['test'])
        sub = os.path.join(d, 'results') if kind == 'base' else os.path.join(d, 'plus', name)
        f = os.path.join(sub, f'{name}_s{k}')
        if not os.path.exists(f + '.json'): return None
        j = json.load(open(f + '.json')); p = np.load(f + '_preds.npz')['probs'].argmax(1)
        pred[te] = p; vals.append(j.get('best_val_f1', np.nan)); fa.append(100 * (p == y[te]).mean())
        if 'energy' in j: eng.append(j['energy'])
    return dict(pred=pred, val=float(np.nanmean(vals)), fa=fa, eng=eng)
def summ(r):
    p = r['pred']; cm = confusion_matrix(y, p, labels=range(4)); mi = cm[MI, MI]
    anym = np.isin(y, [MI, HMI]); pm = np.isin(p, [MI, HMI])
    return dict(acc=100 * (p == y).mean(), ci=wilson(int((p == y).sum()), n), f1=f1_score(y, p, average='macro'), kappa=cohen_kappa_score(y, p),
                fold=f"{np.mean(r['fa']):.2f} +- {np.std(r['fa'], ddof=1):.2f}", mi=int(mi), mi_sens=mi / 30, anymi=(anym & pm).sum() / anym.sum(), spec=(~anym & ~pm).sum() / (~anym).sum())
def mcn(a, b):                                   # a, b: prediction vectors; returns (a right & b wrong, a wrong & b right, exact p)
    ra, rb = a == y, b == y; n10 = int((ra & ~rb).sum()); n01 = int((~ra & rb).sum())
    return n10, n01, (binomtest(min(n10, n01), n10 + n01, .5).pvalue if n10 + n01 else 1.0)

base = {m: load('base', m) for m in ['S1', 'A4', 'A1', 'A2r']}
plus = {c: load('plus', c) for c in CONFIGS}; plus = {c: r for c, r in plus.items() if r is not None}
print('variants with complete 5-fold results:', list(plus))
rows = ["| Model | Pooled acc (%) [95% CI] | Fold mean +- SD | Macro-F1 | kappa | MI found of 30 | Any-MI sens. | Any-MI spec. | Mean val macro-F1 (selection) | vs S1 baseline: variant right-S1 wrong / variant wrong-S1 right, exact McNemar p |", "|---|---|---|---|---|---|---|---|---|---|"]
def row(label, r, ref=None):
    s = summ(r); extra = ''
    if ref is not None:
        a, b, pv = mcn(r['pred'], ref['pred']); extra = f"{a}/{b}, p={pv:.3f}"
    rows.append(f"| {label} | {s['acc']:.2f} [{s['ci'][0]:.1f}-{s['ci'][1]:.1f}] | {s['fold']} | {s['f1']:.3f} | {s['kappa']:.3f} | {s['mi']} | {s['anymi']:.3f} | {s['spec']:.3f} | {r['val']:.3f} | {extra} |")
for m, lab in [('S1', 'S1 baseline (current, v3)'), ('A4', 'A4 matched ANN'), ('A1', 'A1 LW-DN161-like'), ('A2r', 'A2 ResNet50')]:
    if base[m] is not None: row(lab, base[m])
for c, r in plus.items(): row(c + (' (all three changes)' if c == 'P6_full' else ''), r, base['S1'])
md = ["### S1+ variants vs the baseline S1 (5-fold cross-validation, 491 images, same folds)"] + rows
# selection by VALIDATION macro-F1 only
if plus:
    best = max(plus, key=lambda c: plus[c]['val'])
    md.append(f"\nVariant chosen by the highest mean VALIDATION macro-F1: **{best}** ({plus[best]['val']:.3f}); baseline S1 validation macro-F1: {base['S1']['val']:.3f}")
    ps = []
    for m in ['S1', 'A4', 'A1', 'A2r']:
        a, b, pv = mcn(plus[best]['pred'], base[m]['pred']); ps.append((m, a, b, pv))
    order = sorted(range(4), key=lambda i: ps[i][3]); adj = [0] * 4; run = 0
    for rank, i in enumerate(order): run = max(run, min(1.0, ps[i][3] * (4 - rank))); adj[i] = run
    md += [f"\nPaired exact McNemar tests of {best} against each baseline (Holm-adjusted over 4 comparisons)", "| vs | chosen right, other wrong | chosen wrong, other right | p | Holm p |", "|---|---|---|---|---|"]
    md += [f"| {m} | {a} | {b} | {pv:.4f} | {adj[i]:.4f} |" for i, (m, a, b, pv) in enumerate(ps)]
    E = plus[best]['eng']
    if E:
        h = [(E_AC * e['sops']) / (E_MAC * 146240.0) for e in E]; w = [e['e_snn_uJ'] / e['e_ann_uJ'] for e in E]
        md.append(f"\nEstimated energy of {best}: head ratio {np.mean(h):.3f}, whole pipeline ratio {np.mean(w):.3f}, spikes/image {np.mean([e['spikes'] for e in E]):.0f}")
    md.append("\nNOTE: all variants were run on the test folds; the table lists ALL of them. Only the variant with the best validation score is called S1+. A variant that is better on the test folds but not on validation must not be chosen after the fact.")
txt = "\n".join(md); print(txt); open(os.path.join(WORK_DIR, 'report_plus.md'), 'w').write(txt)

variants with complete 5-fold results: ['P1_focal', 'P2_membrane', 'P3_distill', 'P6_full']
### S1+ variants vs the baseline S1 (5-fold cross-validation, 491 images, same folds)
| Model | Pooled acc (%) [95% CI] | Fold mean +- SD | Macro-F1 | kappa | MI found of 30 | Any-MI sens. | Any-MI spec. | Mean val macro-F1 (selection) | vs S1 baseline: variant right-S1 wrong / variant wrong-S1 right, exact McNemar p |
|---|---|---|---|---|---|---|---|---|---|
| S1 baseline (current, v3) | 86.15 [82.8-88.9] | 86.15 +- 3.14 | 0.766 | 0.789 | 12 | 0.776 | 0.939 | 0.908 |  |
| A4 matched ANN | 89.00 [85.9-91.5] | 89.00 +- 4.18 | 0.839 | 0.834 | 21 | 0.836 | 0.941 | 0.908 |  |
| A1 LW-DN161-like | 90.63 [87.7-92.9] | 90.63 +- 1.52 | 0.871 | 0.857 | 23 | 0.853 | 0.965 | 0.904 |  |
| A2 ResNet50 | 92.87 [90.2-94.8] | 92.87 +- 1.91 | 0.904 | 0.892 | 24 | 0.897 | 0.976 | 0.935 |  |
| P1_focal | 87.78 [84.6-90.4] | 87.77 +- 2.73 | 0.788 | 0.814 | 14 | 0.793 | 0.941 | 0.894 | 19/11, p=0.200 |
| P2_membran

2047

In [ ]:
# Cell 23: pack the small result files of the PLUS experiment (no model weights) and download
import os, zipfile
from google.colab import files
zp = '/content/plus_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for base, _, fs in os.walk(WORK_DIR):
        for f in fs:
            if f.endswith(('.md', '.json', '.npz', '.csv')) and 'cache' not in base and ('plus' in base or f.startswith('report_plus')):
                z.write(os.path.join(base, f), os.path.relpath(os.path.join(base, f), WORK_DIR))
print('zip size (MB):', round(os.path.getsize(zp) / 1e6, 1)); files.download(zp)

zip size (MB): 0.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 24: CONTROL - give the matched ANN (A4) the SAME distillation, so that "distillation helps" can be separated from "distillation helps spiking networks"
# Run after Cells 1, 2, 4, 5, 19, 20 (and 20b) in the same session. About 30 minutes on a T4. Resumable.
import os, json, time, numpy as np, torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score
from scipy.stats import binomtest

class _SpikeECGPlus2(SpikeECGPlus):
    def __init__(self, n_cls, K=12, readout='spike'):
        if readout == 'ann':
            SpikeECG.__init__(self, n_cls, K, 'polar_temporal', 'lead_aware', 8, ann=True)    # same network as A4 (ReLU instead of spikes)
        else:
            super().__init__(n_cls, K, readout)
SpikeECGPlus = _SpikeECGPlus2
CONFIGS['A4_KD'] = dict(readout='ann', gamma=0.0, kd=0.5, lam=0.0)     # A4 + the same distillation (alpha 0.5, T 2) as P3, no rate penalty

N_FOLDS, REPEATS = 5, 1
CV_DIR = os.path.join(WORK_DIR, 'cv')
meta_full = json.load(open(os.path.join(CACHE_DIR, 'meta.json'))); y_all = np.array(meta_full['y']); n_all = len(y_all)
def make_fold(rep, k):
    d = os.path.join(CV_DIR, f'r{rep}_f{k}'); os.makedirs(d, exist_ok=True); mp = os.path.join(d, 'meta.json')
    if not os.path.exists(mp):
        tv, te = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=100 + rep).split(np.zeros(n_all), y_all))[k]
        a, b = list(StratifiedKFold(6, shuffle=True, random_state=42).split(np.zeros(len(tv)), y_all[tv]))[0]
        m = dict(meta_full); m.update(train=tv[a].tolist(), val=tv[b].tolist(), test=te.tolist()); json.dump(m, open(mp, 'w'))
    xp = os.path.join(d, 'X.npy')
    if not os.path.exists(xp): os.symlink(os.path.join(CACHE_DIR, 'X.npy'), xp)
    return d

t0 = time.time(); c = 'A4_KD'
for k in range(N_FOLDS):
    d = make_fold(0, k); sd = k
    if os.path.exists(os.path.join(d, 'plus', c, f'{c}_s{sd}.json')): print('fold', k, 'done'); continue
    tdir = os.path.join(d, 'teacher'); os.makedirs(tdir, exist_ok=True); tp = os.path.join(tdir, f'A2r_s{sd}.pt')
    if not os.path.exists(tp):
        print(f'--- teacher ResNet-50 | fold {k} ---'); execute_train_run('A2r', d, tdir, seed=sd, epochs=35)
    teacher, _ = build_model('A2r', len(meta_full['classes']), 12)
    teacher.load_state_dict(torch.load(tp, map_location=device)); teacher.to(device).eval()
    for q in teacher.parameters(): q.requires_grad = False
    out = os.path.join(d, 'plus', c); os.makedirs(out, exist_ok=True)
    print(f'--- {c} | fold {k} | elapsed {(time.time()-t0)/60:.0f} min ---')
    train_plus(c, CONFIGS[c], d, out, sd, teacher=teacher)
    del teacher; torch.cuda.empty_cache(); os.remove(tp)
print('Training finished. Minutes:', round((time.time() - t0) / 60))

# ---- report: A4_KD vs A4 (no distillation), vs S1+ (P3_distill) and vs S1
y = y_all; CL = meta_full['classes']; MI = CL.index('Myocardial Infarction'); HMI = CL.index('History of MI')
def pooled(kind, name):
    pr = np.full(n_all, -1); val = []
    for k in range(N_FOLDS):
        dd = os.path.join(CV_DIR, f'r0_f{k}'); te = np.array(json.load(open(os.path.join(dd, 'meta.json')))['test'])
        b = os.path.join(dd, 'results' if kind == 'base' else os.path.join('plus', name), f'{name}_s{k}')
        pr[te] = np.load(b + '_preds.npz')['probs'].argmax(1); val.append(json.load(open(b + '.json'))['best_val_f1'])
    return pr, float(np.mean(val))
def mcn(a, b):
    ra, rb = a == y, b == y; n10 = int((ra & ~rb).sum()); n01 = int((~ra & rb).sum())
    return n10, n01, (binomtest(min(n10, n01), n10 + n01, .5).pvalue if n10 + n01 else 1.0)
res = {'A4 (no distillation)': pooled('base', 'A4'), 'A4_KD (A4 + distillation)': pooled('plus', 'A4_KD'),
       'S1 (no distillation)': pooled('base', 'S1'), 'P3_distill (S1+)': pooled('plus', 'P3_distill')}
lines = ['### Control: distillation applied to the matched ANN as well (5-fold CV, 491 images)', '| Model | Pooled acc (%) | Macro-F1 | MI found of 30 | Mean val macro-F1 |', '|---|---|---|---|---|']
for nme, (p, v) in res.items():
    cm = confusion_matrix(y, p, labels=range(4)); lines.append(f"| {nme} | {100*(p==y).mean():.2f} | {f1_score(y,p,average='macro'):.3f} | {cm[MI,MI]} | {v:.3f} |")
lines += ['', '| Comparison (first right/second wrong, first wrong/second right, exact McNemar p) | |', '|---|---|']
for a, b in [('A4_KD (A4 + distillation)', 'A4 (no distillation)'), ('P3_distill (S1+)', 'A4_KD (A4 + distillation)'), ('P3_distill (S1+)', 'S1 (no distillation)')]:
    n10, n01, pv = mcn(res[a][0], res[b][0]); lines.append(f'| {a} vs {b} | {n10} / {n01}, p = {pv:.3f} |')
txt = '\n'.join(lines); print(txt); open(os.path.join(WORK_DIR, 'report_plus_control.md'), 'w').write(txt)

--- teacher ResNet-50 | fold 0 ---
[A2r Seed 0] Accuracy: 93.94% | Macro-F1: 0.9101
--- A4_KD | fold 0 | elapsed 3 min ---
[A4_KD seed 0] val-F1 0.931 | test acc 94.95% | macro-F1 0.9028
--- teacher ResNet-50 | fold 1 ---
[A2r Seed 1] Accuracy: 93.88% | Macro-F1: 0.8789
--- A4_KD | fold 1 | elapsed 9 min ---
[A4_KD seed 1] val-F1 0.946 | test acc 90.82% | macro-F1 0.7960
--- teacher ResNet-50 | fold 2 ---
[A2r Seed 2] Accuracy: 90.82% | Macro-F1: 0.9024
--- A4_KD | fold 2 | elapsed 16 min ---
[A4_KD seed 2] val-F1 0.901 | test acc 86.73% | macro-F1 0.8455
--- teacher ResNet-50 | fold 3 ---
[A2r Seed 3] Accuracy: 92.86% | Macro-F1: 0.9155
--- A4_KD | fold 3 | elapsed 22 min ---
[A4_KD seed 3] val-F1 0.938 | test acc 90.82% | macro-F1 0.8983
--- teacher ResNet-50 | fold 4 ---
[A2r Seed 4] Accuracy: 93.88% | Macro-F1: 0.9525
--- A4_KD | fold 4 | elapsed 29 min ---
[A4_KD seed 4] val-F1 0.875 | test acc 87.76% | macro-F1 0.8148
Training finished. Minutes: 30
### Control: distillation appli

708